# Working With `radiosim`, `pyvisgen`, and `pyvisgrid`

`pyvisgen` is a simulation framework that uses the Radio Interferometer Measurment Equation (RIME)

$$\mathbf{V}_{\mathrm{pq}}(l, m) = \sum_{l, m} \mathbf{E}_{\mathrm{p}}(l, m) \mathbf{K}_{\mathrm{p}}(l, m) \mathbf{B}(l, m) \mathbf{K}^{H}_{\mathrm{q}}(l, m) \mathbf{E}^{H}_{\mathrm{q}}(l, m)$$

to simulate the measurement process of a radio
interferometer. The entire framework is written in PyTorch, which allows utilizing the GPU and
improves computation speed multifold over CPU-only approaches. The inputs for the framework are 𝑁 px × 𝑁 px images of sky
distributions.

In the following, we will have a look on how to

1) Simulate (true) sky sources with another package of the [`radionets-project`](https://github.com/radionets-project) called [`radiosim`](https://github.com/radionets-project/radiosim)
2) Simulate observations of those sources using `pyvisgen`
3) Grid and visualise data from `pyvisgen` using `pyvisgrid`

## 1. Simulating Sources With `radiosim`

`radiosim` allows simulating radio skies to create astrophysical datasets. For this tutorial, we will use the [`mojave`](https://radiosim.readthedocs.io/en/latest/api-reference/mojave.html) module of
`radiosim` to create sources which parameters are derived from observational data of the [MOJAVE](https://www.cv.nrao.edu/MOJAVE/index.html) (Monitoring Of Jets in Active galactic nuclei with VLBA Experiments) survey obtained with the Very Long Baseline Array (VLBA).

We start by importing the main function `create_mojave()`:

In [ ]:
from radiosim.mojave import create_mojave
from radiosim.io import Config

`create_mojave(conf, rng)` requires two positional arguments `conf`, and `rng`. The `conf` object is a dictionary containing several keys that provide some settings to the function. `rng` is a [NumPy Random Generator](https://numpy.org/doc/2.3/reference/random/generator.html) instance. As such, we will now import NumPy and create a `default_rng` instance:

In [ ]:
import numpy as np

rng = np.random.default_rng(42)  # The seed 42 is optional, but makes everything reproducible

Now we can also create the `conf` dictionary:

In [ ]:
conf = Config()
conf.general.threads = 1
conf.dataset.img_size=512
conf.dataset.bundle_size=1
conf.mojave.class_ratio= [0, 0, 1]

Breakdown of the configuration:
- `threads` is the number of parallel jobs or threads run during the simulation
- `img_size` specifies the size of the $N\times N$ pixel image
- `bundle_size` is the number of images per bundle. This is only relevant when producing a dataset with multiple images that are saved to multiple files in bundles
- `class_ratio` is the ratio of the three source classes that can be simulated in the `mojave` module:
    - Compact sources: Sources with compact core structures
    - Single-jet sources: Sources with one visible jet extending from the core
    - Double-jet sources: Sources with two opposing jets


We can now pass `conf` and `rng` to `create_mojave()`

In [ ]:
data, data_name = create_mojave(conf, rng)

which returns two objects: `data`, which contains the simulated array, as well as metadata:

In [ ]:
data

And `data_name` which contains the metadata keys corresponding to the data:

In [ ]:
data_name

We now know, that the image data is saved in the first entry of the data list. We can visualize the simulated radio sky using matplotlibs `ax.imshow()` method:

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"image.origin": "lower", "image.cmap": "inferno"})

In [ ]:
sky = data[0]

In [ ]:
sky.shape  # Looking at the shape, we need sky[0] to display the image

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")

im = ax.imshow(sky[0])
fig.colorbar(
    im,
    ax=ax,
    label=r"Flux Density $/\: \mathrm{Jy}\cdot\mathrm{px}^{-1}$",
    shrink=0.92,
)

ax.grid(False)

ax.set(
    xlabel="px",
    ylabel="px"
);

Note, that with the CLI tool of `radiosim`, you can create multiple bundles of simulated radio skies in one go:

```shell-session
$ radiosim-simulate --mode simulate PATH/TO/CONFIG/FILE.toml
```

A typical configuration file for `radiosim` looks like this:

```toml
title = "Simulation configuration"

[general]
quiet = true
seed = "none"  # "none" or int
mode = "mojave"   # jet or survey or mojave
threads = "none" # "none" or int > 0

[paths]
outpath = "./build/example_data/"

[jet]
training_type = "list"  # gauss or list or clean
num_jet_components = [3, 10]
scaling = "normalize"

[survey]
num_sources = 20
# classes are: jet, gaussian, pointlike
class_distribution = [2, 1, 2]   # average share of each source in the images
scale_sources = true

[mojave]
class_ratio = [1, 1, 1] # rel. amount of each class [compact, one_jet, two_jets]

[image_options]
bundles_train = 1
bundles_valid = 1
bundles_test = 1
bundle_size = 100
img_size = 512
noise = true
noise_level = [0, 15]    # range of the intensity of noise in percent, relative to peak flux in image
```

To simulate MOJAVE-like sources, set the `mode` in the `[general]` section to `"mojave"` and adjust the `class_ratio` in the `[mojave]` section to your liking. Then set the number of `threads` in the `[general]` section to the number of threads you want to use; `"none"` will default to 1 thread. Finally set the number of *train*, *valid*, and *test* bundles using the `bundles_train`, `bundles_valid`, and `bundles_test` keys
respectively, as well as the image size (`img_size`) and the `bundle_size`.

This concludes the section on `radiosim` simulations of MOJAVE-like sources.''

## 2. Simulating Observations Using `pyvisgen`

Now that we have one or more source images, we can go on simulating an observation with `pyvisgen`. `pyvisgen` has two core components:

- A class that sets up the observation ([`pyvisgen.simulation.Observation`](https://pyvisgen.readthedocs.io/en/latest/api-reference/simulation/observation.html)), and
- A function that serves as the main simulation loop, [`pyvisgen.simulation.vis_loop`](https://pyvisgen.readthedocs.io/en/latest/api/pyvisgen.simulation.visibility.vis_loop.html#pyvisgen.simulation.visibility.vis_loop)

For now, we will use both inside this notebook to simulate an observation of the source image we created with `radiosim`. Later on, we will have a look at the CLI of `pyvisgen` that allows
simulating large datasets using the terminal and a configuration file.

We start by importing the `Observation` class, which sets up the observation parameters:

In [ ]:
from pyvisgen.simulation import Observation

The `Observation` class accepts a lot of parameters--this is by design, as it is usually supposed to be called internally via the CLI tool of `pyvisgen`:
```python
Observation(
    src_ra: float,
    src_dec: float,
    start_time: datetime,
    scan_duration: int | np.typing.ArrayLike,
    num_scans: int,
    scan_separation: int | np.typing.ArrayLike,
    integration_time: int | np.typing.ArrayLike,
    ref_frequency: float,
    frequency_offsets: list,
    bandwidths: list,
    fov: float,
    image_size: int,
    array_layout: str,
    corrupted: bool,
    device: str,
    dense: bool = False,
    sensitivity_cut: float = 1e-6,
    polarization: str = None,
    pol_kwargs: dict = DEFAULT_POL_KWARGS,
    field_kwargs: dict = DEFAULT_FIELD_KWARGS,
    show_progress: bool = False,
)
```

For this tutorial we will not go through all parameters, but most of them, and those that are necessary for the simulation of an observation:

Source Parameters:
- `src_ra`: Right ascension of the source
- `src_dec`: Declination of the source

Time Parameters:
- `start_time`: Start date of the observation
- `scan_duration`: Duration of each scan in seconds
- `num__scans`: Total number of scans in the observation
- `scan_separation`: Time gap between consecutive scans in seconds; The array will not measure during these times
- `integration_time`: Integration time for measurements in seconds; Defines the time resolution of measurements within each scan

Frequency Parameters:
- `ref_frequency`: Reference frequency in Hz
- `frequency_offsets`: List of frequency offsets in Hz; Used for multiband observations
- `bandwidths`: List of frequency bandwidths in Hz; Defines the width of each frequency channel

Image Parameters:
- `fov`: Field of view in arcseconds
- `image_size`: Size of the output image in pixels ($N\times N$)

Array Configuration:
- `array_layout`: Name of an existing antenna array configuration; Check out [`pyvisgen.layouts.get_array_names()`](https://pyvisgen.readthedocs.io/en/latest/api/pyvisgen.layouts.get_array_names.html#pyvisgen.layouts.get_array_names) to get a list of available arrays

Computation Parameters:
- `device`: PyTorch device for computation; e.g., `'cuda'` or `'cpu'`

Misc. Parameters:
- `show_progress`: If `True`, shows a progress bar

We will now create an `Observation` class object for the VLBA using some arbitrary observation parameters. Let us take the coordinates of galaxy M87 for this tutorial:

In [ ]:
from astropy.coordinates import SkyCoord

radec = SkyCoord.from_name("M87")
print(radec)

src_ra = radec.ra
src_dec = radec.dec

To find the best observation time window, we can use the [`SourceVisibility`](https://radionets-radiotools.readthedocs.io/en/latest/api/radiotools.visibility.SourceVisibility.html#radiotools.visibility.SourceVisibility) class from `radiotools`, which is also part of the `radionets-project`:

In [ ]:
from radiotools.visibility import SourceVisibility

sv = SourceVisibility(
    (src_ra, src_dec),
    "2025-11-06",
    location="vlba",
    print_optimal_date=True
)

sv.plot()

With this, we can get the best start date for our observation:

In [ ]:
obs_start = sv.get_optimal_date()[0]
obs_start

Let us now create the `Observation` class object itself:

In [ ]:
obs = Observation(
    src_ra=src_ra,
    src_dec=src_dec,
    start_time=obs_start,
    scan_duration=400,
    num_scans=20,
    scan_separation=360,
    integration_time=30,
    ref_frequency=15.17600e9,
    frequency_offsets=[0],
    bandwidths=[1.28e8],
    fov=0.075,
    image_size=sky.shape[-1],  # sky has the shape (1, N, N); in this tutorial N = 512
    array_layout="vlba",
    corrupted=False,
    device="cuda",
    show_progress=True,
)

Let us have a look at the `obs` object we just created. The `Observation` class returns a dataclass called `Baselines` that contains information about the $(u, v, w)$ coordinates of the baselines, the corresponding antennas (`st1`, `st2`) and a mask of `valid` baselines, i.e. those that can actually be observed by the antennas.

In [ ]:
bas = obs.baselines
bas

In [ ]:
# Uncomment the line below to see all other attributes of the Observation class
# print(obs.__dict__)

This `Baselines` dataclass is used in the `vis_loop` to simulate the output data of the array. We can now have a look at the $(u,v)$ coverage of the observation:

In [ ]:
# We want only valid u, v coordinates, so we use the valid mask here
# (convert Tensor to boolean tensor first)
u = bas.u[bas.valid.bool()]
v = bas.v[bas.valid.bool()]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(u, v, color="C0", s=12)
ax.scatter(-u, -v, color="C0", s=12)

ax.set(xlabel=r"$u \:/\: \lambda$", ylabel=r"$v \:/\: \lambda$");

This is a typical coverage for an array like the VLBA. The source will only be sparsely covered in $(u, v)$ space, resulting in noise in the Fourier-transformed image.

We will now perform the final step of the simulation using the `vis_loop`:

In [ ]:
from pyvisgen.simulation import vis_loop

In it's simplest form, the `vis_loop` only requires the `Observation` class object `obs`, and the sky image:

In [ ]:
# we have to convert the sky image to a torch.Tensor object:
import torch

_sky = torch.from_numpy(sky)

vis = vis_loop(obs, _sky, show_progress=True)

In [ ]:
vis.V_11.shape

In [ ]:
vis.get_values().shape

In [ ]:
torch.Size([4]) + torch.Size([4])

In [ ]:
vis.get_values()[0], vis.V_11[0]

In [ ]:
batches = torch.arange(vis.base_num.shape[0]).split(1000)

In [ ]:
for p in batches:
    bas_p = bas[p]

In [ ]:
bas[batches[0]].st1.shape

In [ ]:
from dataclasses import fields

In [ ]:
for idx, f in enumerate(fields(vis)):
    print(idx, f.name, getattr(vis, f.name).shape)

In [ ]:
vis.base_num.shape[0]

Let us now have a look at the `vis` object we just created. Analogously to the `Observation` class, the `vis_loop` also returns a dataclass object that is called `Visibilities`.
Among others, the object contains the complex visibilities of the Stokes matrix $\mathbf{B}$, timestamps (`date`), and the $u$, $v$, and $w$ coordinates used for the simulation.

In [ ]:
vis

Again, we can access the respective tensors directly through the dataclass object:

In [ ]:
vis.V_11.shape, vis.V_11

We can also obtain the visibility/coherency matrix $\mathbf{V}$, from the `vis` object by using the `get_values()` method like so:

In [ ]:
vis_data = vis.get_values()
print(vis_data.shape)

vis_data

The result is a tensor with shape (N, M, 4) where:

- N is the number of visibility measurements
- M is the number of frequency channels
- 4 represents the four visibility components `V_11`, `V_22`, `V_12`, `V_21`

Analogously to `radiosim`, `pyvisgen` can also be used through CLI tools. To simulate datasets we can use the `pyvisgen-simulate` tool, e.g.:

```shell-session
$ pyvisgen-simulate --mode gridding -k galaxies -p all PATH/TO/CONFIG/FILE.toml
```

This will create a dataset from, e.g., a `radiosim` dataset for images stored under the `galaxies` key with multiprocessing enabled for all available cores (`-p all`). Further, the `mode` flag allows to grid the data before saving (see next section).

A typical configuration file for `pyvisgen` may look like this:

```toml
[sampling]
mode = "full"
device = "cuda"
seed = 1337
layout = "vlba"
img_size = 1024
fov_center_ra = [100, 110]
fov_center_dec = [30, 40]
fov_size = 0.24
corr_int_time = 30.0
scan_start = ["01-01-1995 00:00:01", "01-01-2025 23:59:59"]
scan_duration = [20, 600]
num_scans = [6, 10]
scan_separation = 360
ref_frequency = 15.17600e9
frequency_offsets = [0e8, 1.28e8, 2.56e8, 3.84e8]
bandwidths = [1.28e8, 1.28e8, 1.28e8, 1.28e8]
noisy = 0
corrupted = false
sensitivity_cut = 1e-6

[polarization]
mode = "none"              # linear, circular, or "none"
delta = 45                 # phase angle
amp_ratio = 0.5            # polarization amplitude ratio
field_order = [0.01, 0.01] # (x, y) orders of the random polarization field
field_scale = [0, 1]       # scaling of the intensity of the polarization field
field_threshold = "none"

[bundle]
dataset_type = "train"  # e.g. train, test, or valid. If "none", no prefix will be applied
in_path = "./path/to/input/data/"
out_path = "./output/path/"
output_writer = "H5Writer"
overlap = 5
grid_size = 1024
grid_fov = 0.24
amp_phase = false

[gridding]
gridder = "default"
```

This will take input data from `./path/to/input/data` (`in_path` in `[bundle]` section), and save it as `HDF5` files (see `"H5Writer"` in the `[bundle]` section) to the output path
`./output/path` (`out_path` in `[bundle]` section). Some of the parameters in the `[sampling]` section will be drawn randomly from a given range to allow for variation in the observational data.

## 3. Gridding and Visualisation With `pyvisgrid` 

To visualise the data we obtained with the `vis` object, we first need to discretize or "grid" it. For that purpose, we have developed the [`pyvisgrid`](https://pyvisgrid.readthedocs.io/en/latest/) package as part of the `radionets-project`. This package also comes with handy plotting functions to help with the visualisation.

In [ ]:
from pyvisgrid.core import Gridder

The `Gridder` class we just imported comes with a classmethod called `from_pyvisgen()` which makes gridding `pyvisgen` data fairly simple; all we need is the visibility data, the `Observation` class object, the image size, and the FOV. (Internally, this method gets the coherency matrix through the same `get_values()` method we explored above)

In [ ]:
gridded_data = Gridder.from_pyvisgen(vis, obs, img_size=512, fov=0.075).grid()

`gridded_data` contains the so-called dirty image, i.e., the uncleaned, Fourier-transformed image that results from the observational data:

In [ ]:
gridded_data

We can plot this data using the `plot_dirty_image()` function of the `pyvisgrid.plotting` module:

In [ ]:
from pyvisgrid.plotting import plot_dirty_image

Real part:

In [ ]:
fig, ax = plot_dirty_image(gridded_data, mode="real")
ax.grid(False)

Imaginary part:

In [ ]:
fig, ax = plot_dirty_image(gridded_data, mode="imag")
ax.grid(False)

Absolute:

In [ ]:
fig, ax = plot_dirty_image(gridded_data, mode="abs")
ax.grid(False)

We can see, that this image is noisy and does not look like the original sky distribution we simulated with `radiosim`. This is because of the sparse sampling of the $(u, v)$ space that we simulated with `pyvisgen`. To clean the image, we will need a cleaning algorithm or a deep-learning approach like `radionets`.


For now, however, this concludes this tutorial. We have learned how we can use
- `radiosim` to simulate a (true) sky distribution
- `pyvisgen` to setup an observation and to simulate the output data of a radio interferometer
- `pyvisgrid` to discretize the obtained data and visualise the results